# EquiNexus final reproduction: Solar Filament Segmentation 2026

This portable notebook runs locally in VS Code/Jupyter or as a Kaggle notebook. It installs the pinned environment, locates the official competition input, optionally reproduces the grouped YOLO11s training stage, loads the packaged model weights, runs three-family inference, validates the masks, and writes the selected CSV.

**No Kaggle account is required to run it locally** when the competition data is already available on the computer. Competition data is not redistributed in this repository. If no local or Kaggle test images are present, `auto` mode downloads and verifies the exact selected CSV from the public GitHub release; this artifact-only path does not run model inference or recheck image coverage.

The final inference path loads the packaged weights and does not train. The optional `RUN_YOLO_OOF_TRAINING` switch reproduces the five-fold EquiNexus YOLO11s development training run from competition training annotations; it is off by default and is not needed to reproduce the selected CSV. The U-Net, YOLO26x, and Mask R-CNN/refiner checkpoints are upstream model weights included in the release archive.

For full inference, provide the authorized competition dataset either at `data/raw/MAGFiLO_1.0_Kaggle_2026/`, through `EQUINEXUS_DATA_ROOT`, or as an attached Kaggle input. Only the supplied test H-alpha images are used for inference; no test labels are read. The notebook records the resolved source commit, dependency versions, hardware, run mode, hashes, runtime, and validation results.


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys
import tempfile
import urllib.request
import zipfile

IS_KAGGLE = Path("/kaggle/input").is_dir() or bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))
RUN_ID = datetime.now().strftime("%Y%m%d-%H%M%S-%f")
SOURCE_REF = os.environ.get("EQUINEXUS_SOURCE_REF", "main")


def locate_repository():
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "requirements.txt").is_file() and (candidate / "scripts" / "reproduce_final_three_family_consensus.py").is_file():
            return candidate
    return None


if IS_KAGGLE:
    WORK_ROOT = Path("/kaggle/working")
    REPO_ROOT = WORK_ROOT / f"EquiNexus-source-{RUN_ID}"
    REPO_ROOT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", SOURCE_REF,
         "https://github.com/BrownAssassin/EquiNexus.git", str(REPO_ROOT)],
        check=True,
    )
else:
    REPO_ROOT = locate_repository()
    if REPO_ROOT is None:
        raise FileNotFoundError(
            "Open this notebook from an EquiNexus checkout or set the working directory to the repository root."
        )
    WORK_ROOT = REPO_ROOT


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


try:
    SOURCE_COMMIT = subprocess.check_output(
        ["git", "rev-parse", "HEAD"], cwd=REPO_ROOT, text=True
    ).strip()
except (OSError, subprocess.CalledProcessError):
    SOURCE_COMMIT = "unavailable"
try:
    SOURCE_WORKTREE_DIRTY = bool(subprocess.check_output(
        ["git", "status", "--porcelain"], cwd=REPO_ROOT, text=True
    ).strip())
except (OSError, subprocess.CalledProcessError):
    SOURCE_WORKTREE_DIRTY = None

RUN_ROOT = REPO_ROOT / "artifacts" / "final-reproduction" / RUN_ID
RUN_ROOT.mkdir(parents=True, exist_ok=False)
CACHE_ROOT = (
    Path("/kaggle/temp/equinexus-cache") if IS_KAGGLE else REPO_ROOT / ".cache" / "equinexus"
)
TEMP_ROOT = (
    Path("/kaggle/temp/equinexus-tmp") if IS_KAGGLE else REPO_ROOT / ".tmp" / "equinexus"
)
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
TEMP_ROOT.mkdir(parents=True, exist_ok=True)
os.environ.update({
    "PIP_CACHE_DIR": str(CACHE_ROOT / "pip"),
    "TORCH_HOME": str(CACHE_ROOT / "torch"),
    "YOLO_CONFIG_DIR": str(CACHE_ROOT / "ultralytics"),
    "TMP": str(TEMP_ROOT),
    "TEMP": str(TEMP_ROOT),
    "TMPDIR": str(TEMP_ROOT),
    "PYTHONPYCACHEPREFIX": str(CACHE_ROOT / "pycache"),
    "PYTHONDONTWRITEBYTECODE": "1",
})
tempfile.tempdir = str(TEMP_ROOT)
print({"runtime": "Kaggle" if IS_KAGGLE else "local", "repository": str(REPO_ROOT),
       "source_ref": SOURCE_REF if IS_KAGGLE else "current checkout",
       "source_commit": SOURCE_COMMIT, "source_worktree_dirty": SOURCE_WORKTREE_DIRTY,
       "run_directory": str(RUN_ROOT)})


## 1. Set up the Python environment

The repository's root `requirements.txt` pins the tested Python packages, including the inference and optional training dependencies. This cell installs from that file in either runtime. In a local setup, create/select the repository `.venv` kernel before running the notebook.


In [ ]:
INSTALL_REQUIREMENTS = True
if INSTALL_REQUIREMENTS:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
         "--no-cache-dir", "-r", str(REPO_ROOT / "requirements.txt")],
        check=True,
    )
else:
    print("Skipped install; ensure the active kernel matches requirements.txt.")


## 2. Locate the competition dataset

For a local run, place an already obtained dataset from the [official competition Data tab](https://www.kaggle.com/competitions/filament-segmentation-2026/data) under `data/raw/MAGFiLO_1.0_Kaggle_2026/`, or set `EQUINEXUS_DATA_ROOT` to its extracted root. The expected layout includes `train/train_images/`, `train/MAGFiLO_1.0_Annotations_kaggle2026_train.json`, and `test/test_images/`. No Kaggle token or CLI login is used by the notebook. When the dataset is absent, `auto` mode uses the hash-verified public submission artifact instead.

In Kaggle, attach `filament-segmentation-2026`; the notebook searches the mounted input directories. To use another mode, set `RUN_MODE` to `"inference"` or `"artifact"` below.


In [ ]:
import torch

RUN_MODE = "auto"  # "auto": infer when data is found, else exact CSV artifact; "inference" or "artifact"
RUN_YOLO_OOF_TRAINING = False  # Optional, expensive five-fold development training
DEVICE = os.environ.get("EQUINEXUS_DEVICE", "0" if torch.cuda.is_available() else "cpu")
EXPECTED_SPLIT_SHA256 = "30cb2ec52437daf97e769a3a61bf83350062e6cf4f5d00af1f8399649065e48e"
EXPECTED_OOF_FOLDS_SHA256 = "f856708222f5747cd35760932f535e4ae2752fd5161b9cd5981fbf4bbde26a3c"
EXPECTED_WEIGHTS_SHA256 = "38c349623e416e1144eb19ed411c815b6d17020f819d56bab446a2bbadbb88d4"
EXPECTED_SUBMISSION_SHA256 = "45688ce9ad0921aaa4fd4206e4fada2e0c62f33b896a7ddc2bdd729c7c28dcf6"
WEIGHTS_URL = "https://github.com/BrownAssassin/EquiNexus/releases/download/filament-final-2026/final-weights.zip"
SUBMISSION_URL = "https://github.com/BrownAssassin/EquiNexus/releases/download/filament-final-2026/selected-submission.csv"

baseline = json.loads((REPO_ROOT / "configs" / "validation-baseline.json").read_text(encoding="utf-8"))
if baseline.get("expected_split_sha256") != EXPECTED_SPLIT_SHA256:
    raise ValueError("Frozen observation split digest changed.")
oof_manifest = REPO_ROOT / "configs" / "observation_oof_seed42_5fold.csv"
if sha256_file(oof_manifest) != EXPECTED_OOF_FOLDS_SHA256:
    raise ValueError("Frozen five-fold OOF manifest hash changed.")
print("verified frozen split and OOF manifest:", EXPECTED_SPLIT_SHA256, EXPECTED_OOF_FOLDS_SHA256)


def looks_like_dataset(path):
    path = Path(path).expanduser()
    return (
        (path / "train" / "MAGFiLO_1.0_Annotations_kaggle2026_train.json").is_file()
        and (path / "train" / "train_images").is_dir()
        and (path / "test" / "test_images").is_dir()
    )


candidates = []
if os.environ.get("EQUINEXUS_DATA_ROOT"):
    candidates.append(Path(os.environ["EQUINEXUS_DATA_ROOT"]))
candidates.append(REPO_ROOT / "data" / "raw" / "MAGFiLO_1.0_Kaggle_2026")
if IS_KAGGLE:
    kaggle_input = Path("/kaggle/input")
    candidates.append(kaggle_input / "competitions" / "filament-segmentation-2026" / "MAGFiLO_1.0_Kaggle_2026")
    candidates.extend(kaggle_input.rglob("MAGFiLO_1.0_Kaggle_2026"))
    candidates.extend(path.parent.parent for path in kaggle_input.rglob("MAGFiLO_1.0_Annotations_kaggle2026_train.json"))
DATA_ROOT = next((path.expanduser().resolve() for path in candidates if looks_like_dataset(path)), None)

if RUN_MODE not in {"auto", "inference", "artifact"}:
    raise ValueError("RUN_MODE must be auto, inference, or artifact")
if RUN_MODE == "auto":
    RUN_MODE = "inference" if DATA_ROOT is not None else "artifact"
if RUN_MODE == "inference" and DATA_ROOT is None:
    raise FileNotFoundError("Inference requires the competition dataset. Set EQUINEXUS_DATA_ROOT or attach the Kaggle input.")

TEST_IMAGE_DIR = DATA_ROOT / "test" / "test_images" if DATA_ROOT else None
if TEST_IMAGE_DIR is not None:
    test_images = sorted(TEST_IMAGE_DIR.glob("*.jpeg"))
    if len(test_images) != 180:
        raise ValueError(f"Expected 180 supplied test images, found {len(test_images)} at {TEST_IMAGE_DIR}")
    print("data root:", DATA_ROOT, "| supplied test images:", len(test_images))
else:
    print("No competition dataset found. Artifact-only mode will restore the exact selected CSV; model inference will not run.")

print({"mode": RUN_MODE, "device": DEVICE, "cuda_available": torch.cuda.is_available(),
       "torch": torch.__version__, "torch_cuda": torch.version.cuda})


## 3. Preprocessing and grouped training data

When training is enabled, the shared loader validates the competition COCO annotation file, image dimensions, image-to-annotation links, and the observation grouping. The frozen seed-42, year-stratified 20% holdout remains excluded; the five OOF folds cover the 565-observation development partition without splitting annotator views across folds. The training script converts the instance annotations into Ultralytics segmentation labels, exports fold-specific image/label directories under ignored `artifacts/`, and trains from the public COCO-initialized YOLO11s-seg checkpoint. It does not train on test images or test labels.

The selected inference run does not load the training JSON. It reads only the 180 supplied test H-alpha images and uses the packaged model weights. Training is an optional, separate reproduction of the YOLO11s OOF development stage.


In [ ]:
if RUN_YOLO_OOF_TRAINING:
    if DATA_ROOT is None:
        raise FileNotFoundError("YOLO OOF training requires the competition training data.")
    sys.path.insert(0, str(REPO_ROOT / "src"))
    from filament_baseline.data import inventory, load_dataset

    dataset = load_dataset(DATA_ROOT)
    print("validated competition training data:", inventory(dataset))
    baseline = json.loads((REPO_ROOT / "configs" / "validation-baseline.json").read_text(encoding="utf-8"))
    if baseline.get("expected_split_sha256") != EXPECTED_SPLIT_SHA256:
        raise ValueError("Frozen observation split digest changed.")
    oof_source = REPO_ROOT / "configs" / "observation_oof_seed42_5fold.csv"
    if sha256_file(oof_source) != EXPECTED_OOF_FOLDS_SHA256:
        raise ValueError("Frozen five-fold OOF manifest hash changed.")
    split_dir = REPO_ROOT / "data" / "splits"
    split_dir.mkdir(parents=True, exist_ok=True)
    oof_target = split_dir / oof_source.name
    if oof_target.exists() and sha256_file(oof_target) != EXPECTED_OOF_FOLDS_SHA256:
        raise ValueError(f"Existing OOF manifest has a different hash: {oof_target}")
    if not oof_target.exists():
        shutil.copy2(oof_source, oof_target)
    training_run_name = f"yolo11s-seg-1280-40ep-oof-{RUN_ID}"
    train_command = [
        sys.executable, str(REPO_ROOT / "scripts" / "run_yolo_training_partition_oof.py"),
        "--repo-root", str(REPO_ROOT), "--data-root", str(DATA_ROOT),
        "--oof-manifest", str(oof_target), "--run-name", training_run_name,
        "--folds", "5", "--model", "yolo11s-seg.pt", "--image-size", "1280",
        "--epochs", "40", "--batch-size", "2", "--workers", "4", "--patience", "12",
        "--device", DEVICE,
    ]
    print("Training command:", " ".join(train_command))
    subprocess.run(train_command, cwd=REPO_ROOT, check=True)
else:
    print("Optional training skipped. The final inference path uses the hash-verified packaged checkpoints without retraining.")


## 4. Acquire and verify the packaged weights

The release archive contains all 18 checkpoints used by the selected three-family ensemble, the per-file SHA-256 manifest, and upstream license notices. A local copy under `data/reference/final-weights-selected-20261114/` is reused after every file is verified. Otherwise, the notebook downloads the public release asset, checks its archive hash, extracts it safely inside the repository's ignored `data/` folder, then verifies each checkpoint. No Kaggle account is used to acquire the weights.


In [ ]:
def download_file(url, destination):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    request = urllib.request.Request(url, headers={"User-Agent": "EquiNexus-reproduction/1.0"})
    with urllib.request.urlopen(request, timeout=180) as response, destination.open("wb") as stream:
        shutil.copyfileobj(response, stream, length=8 * 1024 * 1024)
    return destination


def verify_weights(root):
    manifest_path = Path(root) / "weights-manifest.json"
    if not manifest_path.is_file():
        return False
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    for record in manifest.get("weights", []):
        path = (Path(root) / record["archive_path"]).resolve()
        if Path(root).resolve() not in path.parents or not path.is_file():
            return False
        if path.stat().st_size != int(record["size_bytes"]):
            return False
        if sha256_file(path) != record["sha256"]:
            return False
    return len(manifest.get("weights", [])) == 18


WEIGHTS_ROOT = REPO_ROOT / "data" / "reference" / "final-weights-selected-20261114"
if RUN_MODE == "inference" and not verify_weights(WEIGHTS_ROOT):
    if WEIGHTS_ROOT.exists():
        WEIGHTS_ROOT = REPO_ROOT / "data" / "reference" / f"final-weights-{RUN_ID}"
    archive_path = RUN_ROOT / "final-weights.zip"
    download_file(WEIGHTS_URL, archive_path)
    if sha256_file(archive_path) != EXPECTED_WEIGHTS_SHA256:
        raise ValueError("Final weight archive SHA-256 mismatch.")
    WEIGHTS_ROOT.mkdir(parents=True, exist_ok=False)
    with zipfile.ZipFile(archive_path) as archive:
        for member in archive.infolist():
            target = (WEIGHTS_ROOT / member.filename).resolve()
            if WEIGHTS_ROOT.resolve() not in target.parents and target != WEIGHTS_ROOT.resolve():
                raise ValueError(f"Unsafe archive path: {member.filename}")
        archive.extractall(WEIGHTS_ROOT)
    archive_path.unlink()
    if not verify_weights(WEIGHTS_ROOT):
        raise ValueError("Extracted weight files do not match the weight manifest.")

if RUN_MODE == "inference":
    print("verified checkpoints:", len(json.loads((WEIGHTS_ROOT / "weights-manifest.json").read_text())["weights"]))


## 5. Run the final inference pipeline

For each supplied test image, the pipeline applies the saved ten-model U-Net ensemble and five-fold YOLO11s detector, then runs the HDJoJo YOLO26x model and I079 Mask R-CNN/refiner. It uses the recorded disk normalization, eight-view transforms, confidence/NMS filters, and three-family vote. A candidate instance requires support from at least two families at IoU greater than 0.50; the exactly-two-family case requires each overlap to be at least 0.60. The result is clipped into disjoint instances, assigned unique per-image IDs, and encoded as 2048 x 2048 COCO RLE.

The runner validates coverage of all 180 supplied test images, RLE round trips, nonempty masks, unique IDs, and non-overlapping instances. If a new runtime produces a different CSV hash, the notebook preserves that replay and emits the hash-verified CSV used for the selected Kaggle submission. It reports both hashes so the fallback is explicit.


In [ ]:
REFERENCE_CSV = RUN_ROOT / "selected-submission-reference.csv"
if RUN_MODE == "artifact" or RUN_MODE == "inference":
    download_file(SUBMISSION_URL, REFERENCE_CSV)
    if sha256_file(REFERENCE_CSV) != EXPECTED_SUBMISSION_SHA256:
        raise ValueError("Selected submission asset SHA-256 mismatch.")
    print("verified selected submission asset:", sha256_file(REFERENCE_CSV))

INFERENCE_ROOT = RUN_ROOT / "model-inference"
if RUN_MODE == "inference":
    command = [
        sys.executable, str(REPO_ROOT / "scripts" / "reproduce_final_three_family_consensus.py"),
        "--data-root", str(DATA_ROOT), "--weights-root", str(WEIGHTS_ROOT),
        "--output-root", str(INFERENCE_ROOT), "--device", DEVICE,
    ]
    print("Inference command:", " ".join(command))
    subprocess.run(command, cwd=REPO_ROOT, check=True)
else:
    print("Artifact-only mode selected: model inference is skipped because no competition dataset was found or RUN_MODE was set to artifact.")


## 6. Validate and save the result

Artifact-only mode verifies the published CSV hash, schema, unique instance IDs, 2048 x 2048 RLE decoding and round trips, nonempty masks, and zero within-image overlap. Because it has no test-image directory, it cannot independently recheck the 180-image coverage. Full inference mode uses the repository validator against the supplied test filenames and performs all of those checks.


In [ ]:
import numpy as np
import pandas as pd
from pycocotools import mask as mask_utils


def validate_rles_without_test_images(csv_path):
    frame = pd.read_csv(csv_path, dtype=str, keep_default_na=False)
    expected_columns = ["filament_id", "segmentation_rle"]
    if frame.columns.tolist() != expected_columns or frame.empty:
        raise ValueError(f"Expected nonempty CSV with columns {expected_columns}.")
    if not frame["filament_id"].is_unique:
        raise ValueError("filament_id values are not unique.")
    rows_by_image = {}
    for row in frame.itertuples(index=False):
        parts = row.filament_id.rsplit("_", 1)
        if len(parts) != 2 or not parts[1].isdigit() or int(parts[1]) < 1:
            raise ValueError(f"Malformed filament ID: {row.filament_id}")
        rows_by_image.setdefault(parts[0], []).append((row.filament_id, row.segmentation_rle, int(parts[1])))
    roundtrips = 0
    for image_stem, rows in rows_by_image.items():
        if sorted(item[2] for item in rows) != list(range(1, len(rows) + 1)):
            raise ValueError(f"Non-contiguous instance IDs for {image_stem}")
        claimed = np.zeros((2048, 2048), dtype=np.uint8)
        for filament_id, counts, _ in rows:
            if not counts or not counts.isascii():
                raise ValueError(f"Blank or non-ASCII RLE for {filament_id}")
            decoded = mask_utils.decode({"size": [2048, 2048], "counts": counts.encode("ascii")})
            if decoded.shape != (2048, 2048) or not decoded.any():
                raise ValueError(f"Invalid or empty RLE for {filament_id}")
            encoded = mask_utils.encode(np.asfortranarray(decoded.astype(np.uint8)))
            if encoded["counts"].decode("ascii") != counts:
                raise ValueError(f"RLE round trip changed for {filament_id}")
            if np.any(claimed & decoded):
                raise ValueError(f"Overlapping instances in {image_stem}")
            claimed |= decoded
            roundtrips += 1
    return {"rows": len(frame), "images_with_predictions": len(rows_by_image),
            "rle_shape": [2048, 2048], "rle_exact_roundtrips": roundtrips,
            "unique_ids": True, "all_masks_nonempty": True,
            "instances_overlap_pixels": 0,
            "test_image_coverage_rechecked": False}


if RUN_MODE == "inference":
    replay_csv = INFERENCE_ROOT / "submission.csv"
    if not replay_csv.is_file():
        raise FileNotFoundError(replay_csv)
    replay_hash = sha256_file(replay_csv)
    inference_metadata_path = INFERENCE_ROOT / "reproduction-metadata.json"
    metadata = json.loads(inference_metadata_path.read_text(encoding="utf-8"))
    if replay_hash == EXPECTED_SUBMISSION_SHA256:
        selected_csv = replay_csv
        validation_path = INFERENCE_ROOT / "rle-validation.json"
        validation = json.loads(validation_path.read_text(encoding="utf-8"))
        inference_matches_selected = True
    else:
        shutil.copy2(replay_csv, RUN_ROOT / "inference-replay.csv")
        selected_csv = REFERENCE_CSV
        validation_path = RUN_ROOT / "rle-validation.json"
        subprocess.run([
            sys.executable, str(REPO_ROOT / "scripts" / "validate_submission_rles.py"),
            str(selected_csv), "--test-image-dir", str(TEST_IMAGE_DIR),
            "--expected-test-images", "180", "--output-json", str(validation_path),
        ], cwd=REPO_ROOT, check=True)
        validation = json.loads(validation_path.read_text(encoding="utf-8"))
        metadata["inference_replay_sha256"] = replay_hash
        metadata["inference_matches_selected_submission"] = False
        metadata["selected_submission_source"] = "SHA-256-verified public release asset"
        inference_matches_selected = False
    metadata["inference_replay_sha256"] = replay_hash
    metadata["inference_matches_selected_submission"] = inference_matches_selected
    metadata["selected_submission_sha256"] = EXPECTED_SUBMISSION_SHA256
    metadata["submission_sha256"] = EXPECTED_SUBMISSION_SHA256
    metadata["source_commit_notebook"] = SOURCE_COMMIT
    metadata["source_worktree_dirty"] = SOURCE_WORKTREE_DIRTY
else:
    selected_csv = REFERENCE_CSV
    validation = validate_rles_without_test_images(selected_csv)
    validation_path = RUN_ROOT / "rle-validation.json"
    validation_path.write_text(json.dumps(validation, indent=2) + "\n", encoding="utf-8")
    metadata = {
        "status": "exact submission artifact restored",
        "mode": "artifact-only; no model inference executed",
        "submission_sha256": sha256_file(selected_csv),
        "source_commit_notebook": SOURCE_COMMIT,
        "source_worktree_dirty": SOURCE_WORKTREE_DIRTY,
        "test_images_or_labels_read": False,
        "test_image_coverage_rechecked": False,
        "validation": validation,
    }

if sha256_file(selected_csv) != EXPECTED_SUBMISSION_SHA256:
    raise ValueError("Final selected submission hash mismatch.")

LOCAL_SUBMISSION = RUN_ROOT / "submission.csv"
if selected_csv.resolve() != LOCAL_SUBMISSION.resolve():
    shutil.copy2(selected_csv, LOCAL_SUBMISSION)
metadata_path = RUN_ROOT / "reproduction-metadata.json"
metadata["submission_path"] = str(LOCAL_SUBMISSION)
metadata["submission_sha256"] = EXPECTED_SUBMISSION_SHA256
metadata_path.write_text(json.dumps(metadata, indent=2, sort_keys=True) + "\n", encoding="utf-8")

if IS_KAGGLE:
    kaggle_submission = Path("/kaggle/working/submission.csv")
    shutil.copy2(LOCAL_SUBMISSION, kaggle_submission)
    shutil.copy2(validation_path, Path("/kaggle/working/rle-validation.json"))
    shutil.copy2(metadata_path, Path("/kaggle/working/reproduction-metadata.json"))
    print("submission:", kaggle_submission)
else:
    print("submission:", LOCAL_SUBMISSION)
print(json.dumps({"mode": RUN_MODE, "submission_sha256": sha256_file(LOCAL_SUBMISSION),
                  "validation": validation, "metadata": str(metadata_path)}, indent=2))
